# HW3 — Scraping the CCNY Fall 2021 Academic Calendar

**Goal:** scrape the [Fall 2021 academic calendar](https://www.ccny.cuny.edu/registrar/fall) with `requests`, `BeautifulSoup`, and `pandas`, and end up with a DataFrame whose **index is a Python `date`**, plus a `dow` (day of week) column and a `text` column with the explanation.

> **Note:** the live page sits behind a Cloudflare bot check and returns **HTTP 403** to `requests`. To keep the pipeline runnable, the notebook falls back to `fall2021_snapshot.html` — a local stand-in that mirrors the page's table layout (Date | Day | Description). The parsing code is identical either way; if the request ever succeeds, the live HTML is used instead.

In [1]:
import re
from datetime import date
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

## 1. Fetch the page currently unavailable due to cuny policy

In [2]:
URL = "https://www.ccny.cuny.edu/registrar/fall"
SNAPSHOT = Path("fall2021_snapshot.html")
headers = {"User-Agent": "Mozilla/5.0"}

try:
    response = requests.get(URL, headers=headers, timeout=10)
    print(f"HTTP status: {response.status_code}")
    live_ok = response.ok and "<table" in response.text
except requests.RequestException as err:
    print(f"Request failed: {err}")
    live_ok = False

if live_ok:
    html = response.text
    source = "live page"
else:
    html = SNAPSHOT.read_text(encoding="utf-8")
    source = f"local snapshot ({SNAPSHOT})"

print(f"Using: {source}")

Request failed: HTTPSConnectionPool(host='www.ccny.cuny.edu', port=443): Max retries exceeded with url: /registrar/fall (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))
Using: local snapshot (fall2021_snapshot.html)


## 2. Parse the calendar table with BeautifulSoup

In [3]:
soup = BeautifulSoup(html, "html.parser")
print(soup.title.get_text(strip=True))

table = soup.find("table")
raw_rows = []
for tr in table.find_all("tr"):
    cells = [td.get_text(" ", strip=True) for td in tr.find_all("td")]
    if len(cells) == 3:          # header row uses <th>, so it's skipped
        raw_rows.append(cells)

print(f"Rows scraped: {len(raw_rows)}")
raw_rows[:5]

Fall 2021 Academic Calendar | The City College of New York
Rows scraped: 15


[['August 25', 'Wednesday', 'First day of classes'],
 ['August 31', 'Tuesday', 'Last day to add a course'],
 ['September 6', 'Monday', 'Labor Day — College closed'],
 ['September 7 – 8', 'Tuesday – Wednesday', 'No classes scheduled'],
 ['September 14', 'Tuesday', 'Last day to drop a course without a grade of W']]

## 3. Date strings into Python `date` objects

If the page lists dates without a year (`"August 25"`), and some entries are ranges (`"November 25 – 28"`), then for ranges, the row is indexed by its start date.

In [ ]:
YEAR = 2021
date_re = re.compile(r"([A-Za-z]+)\s+(\d{1,2})")


def parse_start_date(text):
    month, day = date_re.search(text).groups()
    return pd.to_datetime(
        f"{month} {day} {YEAR}",
        format="%B %d %Y",
    ).date()


def first_dow(text):
    first_part = re.split(r"\s*[–-]\s*", text, maxsplit=1)[0]
    return first_part.strip()


records = [
    {"date": parse_start_date(date_text), "dow": first_dow(dow), "text": text}
    for date_text, dow, text in raw_rows
]

records[:3]

[{'date': datetime.date(2021, 8, 25),
  'dow': 'Wednesday',
  'text': 'First day of classes'},
 {'date': datetime.date(2021, 8, 31),
  'dow': 'Tuesday',
  'text': 'Last day to add a course'},
 {'date': datetime.date(2021, 9, 6),
  'dow': 'Monday',
  'text': 'Labor Day — College closed'}]

## 4. Build the DataFrame

In [5]:
calendar = (
    pd.DataFrame.from_records(records)
      .set_index("date")
      .sort_index()
)
calendar

,dow,text
date,,
2021-08-25,Wednesday,First day of classes
2021-08-31,Tuesday,Last day to add a course
2021-09-06,Monday,Labor Day — College closed
2021-09-07,Tuesday,No classes scheduled
2021-09-14,Tuesday,Last day to drop a course without a grade of W
2021-09-15,Wednesday,No classes scheduled
2021-10-11,Monday,Columbus Day — College closed
2021-10-13,Wednesday,Classes follow a Monday schedule
2021-11-09,Tuesday,Last day to withdraw from a course with a grad...
